# Inital Setup
Load the models and configure tokenizer with chat template

In [1]:
import torch
from unsloth.chat_templates import get_chat_template
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Configure BitsAndBytes for 4-bit fast inference
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    "loknezmonzter/qwen-3.5-4b-medquad-lora",
    quantization_config=bnb_config,
    device_map="auto"
)

# Configure tokenizer and apply chat template
tokenizer = AutoTokenizer.from_pretrained("loknezmonzter/qwen-3.5-4b-medquad-lora")
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Model does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


# Evaluation

In [4]:
import re
from datasets import load_dataset

# 1. THE EXTRACTION FUNCTION (Tweak this until it works here!)
def extract_medqa_answer(text):
    # 1. Strip the thinking block if it exists to avoid matching option lists
    clean_text = text.split("</think>")[-1] if "</think>" in text else text
    
    # 2. Look for the explicit final answer format
    match = re.search(r'(?:final answer|answer|choice|option)(?:\s*is)?[\s:]*(ANSWER:\s*[A-D])\b', clean_text, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    
    # 3. Fallback: If no explicit answer, find the last standalone A, B, C, or D 
    # but only in the last 150 characters (the conclusion)
    suffix = clean_text[-150:]
    matches = re.findall(r'\b([A-D])\b', suffix)
    if matches:
        return matches[-1].upper()
        
    return None


# 2. LOAD 2 SAMPLES
test_ds = load_dataset("GBaker/MedQA-USMLE-4-options", split="test", streaming=True)
subset = test_ds.take(5)
samples = list(subset)

# 3. RUN TEST LOOP
val_map = {"A": "0", "B": "1", "C": "2", "D": "3", "0": "0", "1": "1", "2": "2", "3": "3"}

print(f"\033[1;36m=== MANUAL INSPECTION START ===\033[0m\n")

for i, item in enumerate(samples):
    # Format the prompt
    options_str = "\n".join([f"{k}) {v}" for k, v in item["options"].items()])
    prompt_content = f"{item['question']}\n{options_str}\n"

    print(f"\033[1mQuestion:\033[0m {prompt_content}\n")

    messages = [
        {
            "role": "system",
            "content": """
            You are a concise medical evaluator. 
            Complete your reasoning in 3 bullet points. Then, output the final answer.
            The final answer must always be in the below mentioned format: 
            
            ANSWER FORMAT
            --------------
            ANSWER: [LETTER]

            Here are some examples for the answer format:

            EXAMPLE 1
            ----------
            Question:A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?
            A) Ampicillin
            B) Ceftriaxone
            C) Doxycycline
            D) Nitrofurantoin
            REASONING: The patient's symptoms of burning urination and a gravid uterus suggest a urinary tract infection during pregnancy. Nitrofurantoin is a safe and effective treatment for lower UTIs in pregnant women during the second trimester.
            ANSWER: D

            EXAMPLE 2
            ----------
            Question: A 3-month-old baby died suddenly at night while asleep. His mother noticed that he had died only after she awoke in the morning. No cause of death was determined based on the autopsy. Which of the following precautions could have prevented the death of the baby?
            A) Placing the infant in a supine position on a firm mattress while sleeping
            B) Keeping the infant covered and maintaining a high room temperature
            C) Application of a device to maintain the sleeping position
            D) Avoiding pacifier use during sleep
            REASONING: The autopsy revealed no cause of death, indicating Sudden Infant Death Syndrome (SIDS). The most effective preventative measure against SIDS is placing the infant in a supine position on a firm mattress.
            ANSWER: A

            """
        },
        {
            "role": "user",
            "content": prompt_content
        }
    ]
    
    # Inference (Assumes model/tokenizer are already loaded in your notebook)
    inputs = tokenizer.apply_chat_template(
        messages, 
        tokenize=True, 
        add_generation_prompt=True, 
        return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(
        inputs, 
        max_new_tokens=512,
        do_sample=True,
        repetition_penalty=1.0,
        top_p=0.5,
        pad_token_id=tokenizer.eos_token_id
    )
    raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1].strip()
    print(f"\033[1m\nRAW_OUTPUT:\033[0m {raw_output}\n")

    # Apply Debug Regex
    prediction = extract_medqa_answer(raw_output)
    ground_truth = str(item["answer_idx"])
    
    # Compare
    is_correct = val_map.get(prediction) == val_map.get(ground_truth)
    
    # OUTPUT
    print(f"\033[1m--- QUESTION {i+1} ---\033[0m")
    print(f"\033[1;34mGround Truth:\033[0m {ground_truth}")
    print(f"\033[1;34mRegex Extracted:\033[0m {prediction}")
    print(f"\033[1;34mVerdict:\033[0m {'✅ CORRECT' if is_correct else '❌ FAILED'}")
    print(f"\n\033[1;33mRaw Model Reasoning:\033[0m")
    print(f"{raw_output}")
    print("-" * 50 + "\n")

=== MANUAL INSPECTION START ===

Question: A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?
A) Disclose the error to the patient and put it in the operative report
B) Tell the attending that he cannot fail to disclose this mistake
C) Report the physician to the ethics committee
D) Refuse to dictate the operative report



RAW_OUTPUT: A) Disclose the error to the patient and put it in the operative report

--- QUESTION 1 ---
Ground Truth

In [4]:
tokenizer.add_eos_token()

TypeError: 'bool' object is not callable